In [ ]:
!pip install -q ultralytics==8.4.19 sahi ensemble-boxes

import os
from pathlib import Path
import json
import shutil
from typing import List, Dict

import numpy as np
import pandas as pd
from tqdm import tqdm

from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from ensemble_boxes import weighted_boxes_fusion


In [ ]:
DATA_YAML = "/kaggle/working/data_fixed.yaml"
TEST_DIR = "/kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images"
SAMPLE_SUB = "/kaggle/input/competitions/dl-lab-2-stuff-detection/sample_sub.csv"


WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(exist_ok=True, parents=True)

CHECKPOINTS_DIR = WORK_DIR / "checkpoints"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

RUNS_DETECT_DIR = WORK_DIR / "runs" / "detect"
RUNS_DETECT_DIR.mkdir(parents=True, exist_ok=True)

ENSEMBLE_TXT_DIR = WORK_DIR / "ensemble_txt"
ENSEMBLE_TXT_DIR.mkdir(parents=True, exist_ok=True)

SAHI_TXT_DIR = WORK_DIR / "sahi_txt"
SAHI_TXT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from pathlib import Path
import yaml

orig_yaml_path = Path("/kaggle/input/competitions/dl-lab-2-stuff-detection/data.yaml")
with orig_yaml_path.open() as f:
    data_cfg = yaml.safe_load(f)

print("ORIGINAL YAML:", data_cfg)

data_cfg["path"] = "/kaggle/input/competitions/dl-lab-2-stuff-detection/yolo_dataset/yolo_dataset"
data_cfg["train"] = "train/images"
data_cfg["val"] = "train/images"

fixed_yaml_path = Path("/kaggle/working/data_fixed.yaml")
with fixed_yaml_path.open("w") as f:
    yaml.safe_dump(data_cfg, f)

print("Saved fixed yaml to:", fixed_yaml_path)
print("NEW YAML:", data_cfg)



ORIGINAL YAML: {'names': {0: 'customer', 1: 'staff'}, 'path': '/kaggle/input/dl-lab-2-stuff-detection/yolo_dataset/yolo_dataset', 'train': 'train/images', 'val': 'train/images'}
Saved fixed yaml to: /kaggle/working/data_fixed.yaml
NEW YAML: {'names': {0: 'customer', 1: 'staff'}, 'path': '/kaggle/input/competitions/dl-lab-2-stuff-detection/yolo_dataset/yolo_dataset', 'train': 'train/images', 'val': 'train/images'}


In [34]:
import os

print("Listing /kaggle/input:")
print(os.listdir("/kaggle/input"))

print("\nListing /kaggle/input/competitions:")
print(os.listdir("/kaggle/input/competitions"))

print("\nListing /kaggle/input/competitions/dl-lab-2-stuff-detection:")
print(os.listdir("/kaggle/input/competitions/dl-lab-2-stuff-detection"))



Listing /kaggle/input:
['competitions']

Listing /kaggle/input/competitions:
['dl-lab-2-stuff-detection']

Listing /kaggle/input/competitions/dl-lab-2-stuff-detection:
['yolo_dataset', 'sample_sub.csv', 'data.yaml', 'test_images']


In [ ]:
def train_with_resume(
    ckpt_name: str,
    base_weights: str,
    project_name: str,
    run_name: str,
    first_epochs: int,
    extra_epochs: int,
    imgsz: int,
    batch: int,
):

    project_path = WORK_DIR / project_name
    project_path.mkdir(parents=True, exist_ok=True)

    ckpt_path = CHECKPOINTS_DIR / ckpt_name

    if ckpt_path.exists():
        print(f"Found checkpoint {ckpt_path}, continuing training...")
        model = YOLO(str(ckpt_path))
        model.train(
            data=DATA_YAML,
            epochs=extra_epochs,
            imgsz=imgsz,
            batch=batch,
            project=str(project_path),
            name=run_name + "_cont",
            save_period=5,
        )
        best = project_path / (run_name + "_cont") / "weights" / "best.pt"
        if best.exists():
            shutil.copy2(best, ckpt_path)
    else:
        print(f"No checkpoint {ckpt_path}, training from pretrained {base_weights}...")
        model = YOLO(base_weights)
        model.train(
            data=DATA_YAML,
            epochs=first_epochs,
            imgsz=imgsz,
            batch=batch,
            project=str(project_path),
            name=run_name,
            save_period=5,
        )
        best = project_path / run_name / "weights" / "best.pt"
        if best.exists():
            shutil.copy2(best, ckpt_path)

    print(f"Final checkpoint stored at {ckpt_path}")
    return ckpt_path


In [ ]:
# Модель 1: yolo26n (baseline)
ckpt1 = train_with_resume(
    ckpt_name="yolo26n_best.pt",
    base_weights="yolo26n.pt",
    project_name="miet",
    run_name="lab2_yolo26n",
    first_epochs=25,
    extra_epochs=10,
    imgsz=640,
    batch=16,
)

# Модель 2: yolov8m
ckpt2 = train_with_resume(
    ckpt_name="yolov8m_best.pt",
    base_weights="yolov8m.pt",
    project_name="miet",
    run_name="lab2_yolov8m",
    first_epochs=40,
    extra_epochs=10,
    imgsz=768,
    batch=16,
)

# Модель 3: yolov8l
ckpt3 = train_with_resume(
    ckpt_name="yolov8l_best.pt",
    base_weights="yolov8l.pt",
    project_name="miet",
    run_name="lab2_yolov8l",
    first_epochs=40,
    extra_epochs=10,
    imgsz=768,
    batch=8,
)


No checkpoint /kaggle/working/checkpoints/yolo26n_best.pt, training from pretrained yolo26n.pt...
New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_fixed.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_de

In [ ]:
def predict_and_save_txt(model_path: str, name: str, imgsz: int = 768) -> Path:
    model = YOLO(model_path)
    results = model.predict(
        source=TEST_DIR,
        imgsz=imgsz,
        save_txt=True,
        save_conf=True,
        project=str(RUNS_DETECT_DIR),
        name=name,
        stream=True,
    )
    # прогоняем весь стрим, чтобы реально выполнить предсказания
    for _ in results:
        pass

    labels_dir = RUNS_DETECT_DIR / name / "labels"
    pred_dirs.append(labels_dir)
    return labels_dir

In [ ]:
labels1 = predict_and_save_txt(
    "/kaggle/working/miet/lab2_yolo26n5/weights/best.pt",
    "pred_yolo26n",
    imgsz=640,
)


image 1/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-00-00-000.jpg: 384x640 1 customer, 12.1ms
image 2/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-00-12-500.jpg: 384x640 2 customers, 10.0ms
image 3/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-00-25-000.jpg: 384x640 2 customers, 10.4ms
image 4/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-00-37-500.jpg: 384x640 3 customers, 10.3ms
image 5/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-00-50-000.jpg: 384x640 2 customers, 10.4ms
image 6/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-01-02-500.jpg: 384x640 2 customers, 10.2ms
image 7/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-01-15-000.jpg: 384x640 1 customer, 11.1ms
image 8/4454 /kaggle/input/c

In [ ]:
labels2 = predict_and_save_txt(
    "/kaggle/working/miet/lab2_yolov8m/weights/best.pt",
    "pred_yolov8m",
    imgsz=768,
)


image 1/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-00-00-000.jpg: 448x768 1 customer, 16.1ms
image 2/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-00-12-500.jpg: 448x768 2 customers, 15.3ms
image 3/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-00-25-000.jpg: 448x768 2 customers, 15.5ms
image 4/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-00-37-500.jpg: 448x768 2 customers, 15.2ms
image 5/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-00-50-000.jpg: 448x768 2 customers, 15.3ms
image 6/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-01-02-500.jpg: 448x768 3 customers, 14.0ms
image 7/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-01-15-000.jpg: 448x768 1 customer, 14.0ms
image 8/4454 /kaggle/input/c

In [ ]:
labels3 = predict_and_save_txt(
    "/kaggle/working/miet/lab2_yolov8l/weights/best.pt",
    "pred_yolov8l",
    imgsz=768,
)


image 1/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-00-00-000.jpg: 448x768 1 customer, 23.2ms
image 2/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-00-12-500.jpg: 448x768 2 customers, 21.9ms
image 3/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-00-25-000.jpg: 448x768 2 customers, 21.9ms
image 4/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-00-37-500.jpg: 448x768 2 customers, 21.8ms
image 5/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-00-50-000.jpg: 448x768 2 customers, 21.9ms
image 6/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-01-02-500.jpg: 448x768 2 customers, 1 staff, 20.0ms
image 7/4454 /kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images/1.1_00-01-15-000.jpg: 448x768 1 customer, 19.9ms
image 8/4454 /kaggl

In [ ]:
from sahi.model import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from tqdm import tqdm
from pathlib import Path

SAHI_MODEL_PATH = "/kaggle/working/miet/lab2_yolov8m/weights/best.pt"

detection_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",
    model_path=SAHI_MODEL_PATH,
    confidence_threshold=0.2,
    device="cuda:0",
)

def run_sahi_on_test(
    detection_model,
    test_dir: str,
    txt_out_dir: Path,
    slice_height: int = 512,
    slice_width: int = 512,
    overlap_height_ratio: float = 0.2,
    overlap_width_ratio: float = 0.2,
):
    txt_out_dir.mkdir(parents=True, exist_ok=True)
    test_dir = Path(test_dir)
    img_paths = sorted(list(test_dir.glob("*.jpg")))
    for img_path in tqdm(img_paths, desc="SAHI predicting"):
        res = get_sliced_prediction(
            image=str(img_path),
            detection_model=detection_model,
            slice_height=slice_height,
            slice_width=slice_width,
            overlap_height_ratio=overlap_height_ratio,
            overlap_width_ratio=overlap_width_ratio,
        )
        h, w = res.image.shape[:2]

        lines = []
        for obj in res.object_prediction_list:
            cls = obj.category.id
            score = float(obj.score.value)
            xmin, ymin, xmax, ymax = obj.bbox.to_xyxy()

            xc = ((xmin + xmax) / 2.0) / w
            yc = ((ymin + ymax) / 2.0) / h
            bw = (xmax - xmin) / w
            bh = (ymax - ymin) / h
            lines.append(f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f} {score:.6f}")

        out_file = txt_out_dir / f"{img_path.stem}.txt"
        out_file.write_text("\n".join(lines), encoding="utf-8")

run_sahi_on_test(detection_model, TEST_DIR, SAHI_TXT_DIR)
pred_dirs.append(SAHI_TXT_DIR)


ModuleNotFoundError: No module named 'sahi.model'

In [ ]:
from pathlib import Path
from tqdm import tqdm
from ensemble_boxes import weighted_boxes_fusion

def read_yolo_txt(path: Path):
    if not path.exists():
        return [], [], []
    txt = path.read_text(encoding="utf-8").strip()
    if not txt:
        return [], [], []
    boxes = []
    scores = []
    labels = []
    for ln in txt.splitlines():
        parts = ln.strip().split()
        if len(parts) < 6:
            continue
        cls = int(float(parts[0]))
        xc = float(parts[1])
        yc = float(parts[2])
        w = float(parts[3])
        h = float(parts[4])
        sc = float(parts[5])

        x1 = xc - w / 2.0
        y1 = yc - h / 2.0
        x2 = xc + w / 2.0
        y2 = yc + h / 2.0
        boxes.append([x1, y1, x2, y2])
        scores.append(sc)
        labels.append(cls)
    return boxes, scores, labels

def write_yolo_txt(path: Path, boxes, scores, labels):
    lines = []
    for b, s, l in zip(boxes, scores, labels):
        x1, y1, x2, y2 = b
        xc = (x1 + x2) / 2.0
        yc = (y1 + y2) / 2.0
        w = x2 - x1
        h = y2 - y1
        lines.append(f"{int(l)} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f} {float(s):.6f}")
    path.write_text("\n".join(lines), encoding="utf-8")


test_imgs = sorted(Path(TEST_DIR).glob("*.jpg"))

ENSEMBLE_TXT_DIR.mkdir(parents=True, exist_ok=True)

for img_path in tqdm(test_imgs, desc="Ensembling"):
    all_boxes = []
    all_scores = []
    all_labels = []

    for pdir in pred_dirs:
        txt_path = pdir / f"{img_path.stem}.txt"
        b, s, l = read_yolo_txt(txt_path)
        all_boxes.append(b)
        all_scores.append(s)
        all_labels.append(l)

    if not any(len(b) for b in all_boxes):
        out_file = ENSEMBLE_TXT_DIR / f"{img_path.stem}.txt"
        out_file.write_text("", encoding="utf-8")
        continue

    boxes_wbf, scores_wbf, labels_wbf = weighted_boxes_fusion(
        all_boxes,
        all_scores,
        all_labels,
        iou_thr=0.5,
        skip_box_thr=0.001,
    )

    out_file = ENSEMBLE_TXT_DIR / f"{img_path.stem}.txt"
    write_yolo_txt(out_file, boxes_wbf, scores_wbf, labels_wbf)




Ensembling: 100%|██████████| 4454/4454 [00:20<00:00, 218.20it/s]


In [ ]:
from pathlib import Path
import json
import pandas as pd

def build_submission_from_solution_order(
    solution_csv: str,
    preds_dir: str,
    output_csv: str = "submission.csv",
    image_col: str = "image_name",
    boxes_col: str = "boxes",
    row_id_col: str = "id",
    require_score: bool = True,
    clamp_score: bool = True,
    keep_only_class: int | None = None,
) -> None:
    sol_path = Path(solution_csv)
    pdir = Path(preds_dir)

    if not sol_path.exists():
        raise FileNotFoundError(f"solution_csv not found: {sol_path}")
    if not pdir.exists() or not pdir.is_dir():
        raise FileNotFoundError(f"preds_dir not found or not a dir: {pdir}")

    sol = pd.read_csv(sol_path)
    if image_col not in sol.columns:
        raise ValueError(f"solution.csv must contain column '{image_col}'")

    image_names = sol[image_col].astype(str).tolist()

    rows = []
    for idx, image_name in enumerate(image_names):
        stem = Path(image_name).stem
        pred_file = pdir / f"{stem}.txt"

        boxes = []
        if pred_file.exists():
            content = pred_file.read_text(encoding="utf-8", errors="replace").strip()
            if content:
                for ln in content.splitlines():
                    ln = ln.strip()
                    if not ln:
                        continue
                    parts = ln.split()
                    if require_score and len(parts) < 6:
                        continue
                    if len(parts) < 5:
                        continue
                    try:
                        cls = int(float(parts[0]))
                        if keep_only_class is not None and cls != keep_only_class:
                            continue
                        xc = float(parts[1])
                        yc = float(parts[2])
                        w = float(parts[3])
                        h = float(parts[4])
                        sc = float(parts[5]) if len(parts) >= 6 else 1.0
                    except ValueError:
                        continue
                    if clamp_score:
                        sc = 0.0 if sc < 0.0 else (1.0 if sc > 1.0 else sc)
                    boxes.append([xc, yc, w, h, sc])

        rows.append(
            {
                row_id_col: idx,
                image_col: image_name,
                boxes_col: json.dumps(boxes, separators=(",", ":")),
            }
        )

    sub = pd.DataFrame(rows, columns=[row_id_col, image_col, boxes_col])
    sub.to_csv(output_csv, index=False)
    print(f"Saved: {output_csv} ({len(sub)} rows).")


build_submission_from_solution_order(
    solution_csv=SAMPLE_SUB,
    preds_dir=str(ENSEMBLE_TXT_DIR),
    output_csv="submission.csv",
    keep_only_class=1,
)


Saved: submission.csv (4454 rows).
